# Labwork 1 - PCA: Raisin dataset
**Họ tên:** ...  |  **MSSV:** ...

Notebook này làm cả Phần 1 (Study the dataset) và Phần 2 (PCA) cho dataset **Raisin** (UCI id 850).
Sau mỗi đoạn code có một ô *Giải thích* để dễ theo dõi.

## 0. Chuẩn bị

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style="whitegrid")

**Giải thích:** `pandas`/`numpy` xử lý dữ liệu, `matplotlib`/`seaborn` vẽ biểu đồ, `StandardScaler` chuẩn hóa, `PCA` là thuật toán chính của bài.

## 1. Giới thiệu dataset

In [ ]:
raisin = fetch_ucirepo(id=850)          # tải dữ liệu Raisin từ UCI
X = raisin.data.features                # các cột đặc trưng
y = raisin.data.targets                 # cột nhãn (Class)
df = pd.concat([X, y], axis=1)          # gộp thành 1 bảng

df.to_csv("../data/raw/raisin_raw.csv", index=False)   # lưu bản gốc
print(df.shape)
df.head()

**Giải thích:** `fetch_ucirepo(id=850)` tải dữ liệu tự động. Ta gộp features và nhãn thành một bảng `df`, rồi lưu bản gốc vào `data/raw/`.

**Mô tả (điền thêm):** Raisin gồm 900 mẫu của hai giống nho khô (Kecimen và Besni), mỗi mẫu có 7 đặc trưng hình dạng đo từ ảnh. *(TODO: viết 2-3 câu mô tả bằng lời của bạn, trích nguồn UCI.)*

## 2. Loại của từng feature

In [ ]:
print(df.dtypes)

**Giải thích:** `dtypes` cho biết kiểu dữ liệu từng cột.

**TODO - điền bảng phân loại:**

| Feature | Rời rạc / Liên tục | Định lượng / Định tính | Numerical / Categorical |
|---|---|---|---|
| Area | ? | ? | ? |
| MajorAxisLength | ? | ? | ? |
| MinorAxisLength | ? | ? | ? |
| Eccentricity | ? | ? | ? |
| ConvexArea | ? | ? | ? |
| Extent | ? | ? | ? |
| Perimeter | ? | ? | ? |
| Class | ? | ? | ? |

## 3. Vấn đề chất lượng dữ liệu

In [ ]:
print("Missing values:")
print(df.isna().sum())
print()
print("Số dòng trùng lặp:", df.duplicated().sum())
df.describe()

**Giải thích:** `isna().sum()` đếm giá trị thiếu mỗi cột; `duplicated().sum()` đếm dòng trùng; `describe()` cho min/max/mean để phát hiện giá trị bất thường (ví dụ số âm ở cột diện tích).

In [ ]:
# Phát hiện outlier bằng quy tắc IQR
num_cols = df.select_dtypes(include="number").columns

outlier_count = {}
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_count[col] = ((df[col] < low) | (df[col] > high)).sum()

pd.Series(outlier_count, name="so_outlier")

**Giải thích:** một giá trị bị coi là outlier nếu nằm ngoài khoảng [Q1 - 1.5·IQR, Q3 + 1.5·IQR]. Vòng `for` đếm số outlier của từng cột.

**TODO - thảo luận:** các vấn đề tìm thấy (nếu không có missing/duplicate, nói rõ "đã kiểm tra, không có") và chúng ảnh hưởng thế nào đến phân tích (outlier kéo lệch mean/variance, ảnh hưởng PCA...).

In [ ]:
# Boxplot để nhìn outlier
df[num_cols].plot(kind="box", subplots=True, layout=(2, 4), figsize=(14, 7))
plt.tight_layout()
plt.savefig("../figures/raisin_boxplot_before.png", dpi=150)
plt.show()

**Giải thích:** mỗi hộp là một feature, các chấm nằm ngoài là outlier. Ảnh được lưu vào `figures/` để chèn vào báo cáo.

## 4. Làm sạch dữ liệu

In [ ]:
df_clean = df.copy()

# 4.1 Xóa dòng trùng lặp
df_clean = df_clean.drop_duplicates()

# 4.2 Xóa dòng thiếu dữ liệu (nếu có)
df_clean = df_clean.dropna()

# 4.3 Xử lý outlier: ở đây chỉ GIỚI HẠN (clip) về biên IQR thay vì xóa mẫu
for col in num_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    df_clean[col] = df_clean[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

print("Trước:", df.shape, "| Sau:", df_clean.shape)
df_clean.to_csv("../data/processed/raisin_clean.csv", index=False)

**Giải thích:** từng bước xử lý một vấn đề. `clip` giữ nguyên số mẫu nhưng kéo outlier về biên, an toàn khi dataset nhỏ (900 mẫu).

**TODO:** so sánh trước/sau (số dòng, mean, std của vài feature) và giải thích vì sao chọn clip thay vì xóa.

In [ ]:
# So sánh thống kê trước và sau làm sạch
comparison = pd.DataFrame({
    "mean_before": df[num_cols].mean(),
    "mean_after": df_clean[num_cols].mean(),
    "std_before": df[num_cols].std(),
    "std_after": df_clean[num_cols].std(),
})
comparison

## 5. Data integration
**TODO:** Raisin là một bảng duy nhất, không có nguồn khác đi kèm. Viết 3-4 câu giải thích vì sao tích hợp thêm dữ liệu là không cần thiết (hoặc nếu bạn tìm được nguồn bổ sung thì mô tả cách ghép và khó khăn gặp phải).

## 6. Data transformation

In [ ]:
features = df_clean.drop(columns=["Class"])

# Chuẩn hóa: mỗi feature có mean = 0, std = 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# Mã hóa nhãn: chữ -> số
labels = df_clean["Class"].astype("category")
y_encoded = labels.cat.codes
print(dict(enumerate(labels.cat.categories)))

pd.DataFrame(X_scaled, columns=features.columns).describe().round(2)

**Giải thích:** các feature như Area (hàng chục nghìn) và Eccentricity (0-1) khác thang đo rất nhiều, nên phải *standardize* trước khi PCA. Nhãn `Class` là chữ nên được mã hóa thành 0/1.

**TODO:** giải thích vì sao chọn standardization thay vì min-max.

## 7. Label và loại bài toán
**TODO:** nhãn là `Class` (Kecimen / Besni): loại nho khô. Đây là bài toán **phân loại nhị phân (binary classification)** có giám sát. Viết thêm ý nghĩa thực tế.

In [ ]:
df_clean["Class"].value_counts()

**Giải thích:** đếm số mẫu mỗi lớp để xem dữ liệu có cân bằng không.

## 8. Mean, variance, covariance, correlation

In [ ]:
stats = pd.DataFrame({
    "mean": features.mean(),
    "variance": features.var(),
})
print(stats)

print("\nCovariance matrix:")
cov = features.cov()
print(cov.round(2))

corr = features.corr()

**Giải thích:** `var()` là phương sai, `cov()` là ma trận hiệp phương sai, `corr()` là ma trận tương quan Pearson (đã chuẩn hóa về -1...1 nên dễ so sánh).

**Về feature categorical:** Raisin chỉ có 1 cột categorical là `Class`. TODO: nêu cách phân tích phù hợp (bảng tần suất, hoặc so sánh mean của feature theo từng lớp bằng boxplot / t-test).

## 9. Cặp feature tương quan mạnh nhất

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Raisin - correlation matrix")
plt.tight_layout()
plt.savefig("../figures/raisin_corr_heatmap.png", dpi=150)
plt.show()

# Tìm cặp mạnh nhất (bỏ đường chéo)
pairs = corr.abs().where(~np.eye(len(corr), dtype=bool)).stack()
best_pair = pairs.idxmax()
print("Cặp mạnh nhất:", best_pair, "-> r =", round(corr.loc[best_pair], 3))

**Giải thích:** heatmap màu đỏ = tương quan dương, xanh = âm. `where(~np.eye(...))` loại đường chéo (mỗi feature luôn tương quan 1 với chính nó), `idxmax()` lấy cặp lớn nhất.

**TODO:** nhận xét heatmap (nhóm feature nào tương quan cao, vì sao, ví dụ Area và ConvexArea đều đo kích thước nên gần như tuyến tính).

## 10. Tổng kết Phần 1
**TODO:** tóm tắt các phát hiện chính, và việc làm sạch / tích hợp / biến đổi ảnh hưởng thế nào đến chất lượng và độ tin cậy của phân tích.

# Phần 2: PCA

## 11. Áp dụng PCA

In [ ]:
pca = PCA()                     # giữ tất cả thành phần
X_pca = pca.fit_transform(X_scaled)

print("Số thành phần:", X_pca.shape[1])

**Giải thích:** dữ liệu đã standardize (bước tiền xử lý bắt buộc). PCA tìm các hướng (principal components) mà dữ liệu biến thiên nhiều nhất: về bản chất là các eigenvector của ma trận hiệp phương sai, sắp theo eigenvalue giảm dần.

**TODO:** viết 3-4 câu giải thích cách PC được tính (hiệp phương sai → eigenvalue/eigenvector) và cách chọn.

## 12. Explained variance

In [ ]:
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)

table = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(evr))],
    "explained_variance_ratio": evr.round(4),
    "cumulative": cum.round(4),
})
print(table)

plt.figure(figsize=(7, 4))
plt.bar(range(1, len(evr) + 1), evr)
plt.plot(range(1, len(evr) + 1), cum, marker="o", color="red")
plt.xlabel("Principal component")
plt.ylabel("Explained variance ratio")
plt.title("Raisin - explained variance")
plt.tight_layout()
plt.savefig("../figures/raisin_explained_variance.png", dpi=150)
plt.show()

**Giải thích:** `explained_variance_ratio_` là tỉ lệ thông tin (phương sai) mỗi PC giữ lại. Đường đỏ là phương sai tích lũy.

**TODO:** so sánh PC có variance cao nhất với PC thấp nhất: PC đầu giữ cấu trúc chính, PC cuối thường chủ yếu là nhiễu.

## 13. Thay đổi số thành phần

In [ ]:
for k in range(1, X_scaled.shape[1] + 1):
    p = PCA(n_components=k)
    Xk = p.fit_transform(X_scaled)
    X_back = p.inverse_transform(Xk)                 # tái tạo lại dữ liệu
    error = np.mean((X_scaled - X_back) ** 2)        # sai số tái tạo
    print(f"k={k}: giữ {p.explained_variance_ratio_.sum():.1%} variance, sai số tái tạo = {error:.4f}")

**Giải thích:** với mỗi số thành phần k ta xem giữ được bao nhiêu % thông tin, và sai số khi dựng lại dữ liệu gốc từ k thành phần.

**TODO:** nhận xét k nhỏ nhất đạt ví dụ 95% variance và đánh đổi giữa giảm chiều và mất thông tin.

## 14. Trực quan hóa 2D (PC1 vs PC2)

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df_clean["Class"].values, alpha=0.7)
plt.xlabel(f"PC1 ({evr[0]:.1%})")
plt.ylabel(f"PC2 ({evr[1]:.1%})")
plt.title("Raisin - PCA 2D")
plt.tight_layout()
plt.savefig("../figures/raisin_pca_2d.png", dpi=150)
plt.show()

**Giải thích:** mỗi điểm là một mẫu, màu theo `Class`. Nếu hai màu tách nhau thì PC1/PC2 đã nắm được thông tin phân biệt.

**TODO:** nhận xét cụm, vùng chồng lấn.

In [ ]:
# Loadings: mỗi feature đóng góp bao nhiêu vào PC1, PC2 (giúp diễn giải)
loadings = pd.DataFrame(pca.components_[:2].T, index=features.columns, columns=["PC1", "PC2"])
loadings.round(3)

**Giải thích:** loadings cho biết PC là tổ hợp tuyến tính của feature nào, dùng để bàn về khả năng diễn giải.

## 15. So sánh dữ liệu gốc và dữ liệu PCA
**TODO:** thảo luận
- **Ưu điểm:** giảm chiều, loại bỏ tương quan giữa các feature, dễ trực quan hóa.
- **Hạn chế:** mất một phần thông tin, PC khó diễn giải hơn feature gốc, chỉ nắm quan hệ tuyến tính, nhạy với outlier và thang đo.